# SheCraft chunking experiments

Compare four chunking methods across all jewelry PDFs, then evaluate top-five retrieval with a TF-IDF baseline and the provisional SheCraft question set. This immediately tests which chunks match the questions lexically. Later embedding experiments can compare semantic retrieval. The cases and results are for notebook technique selection, not final project evaluation.

In [1]:
import importlib
import json
import sys
from pathlib import Path
from shutil import which

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for folder in (cwd, *cwd.parents)
        for candidate in (folder, folder / "ai-service")
        if (candidate / "src" / "rag" / "doc_rag").is_dir()
        and (candidate / "rag_documents").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from the SheCraft Jewelry workspace.")

SRC_DIR = PROJECT_ROOT / "src" / "rag"
DOCUMENTS_DIR = PROJECT_ROOT / "rag_documents"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Notebook 1 found that this file contains only a title and date.
TITLE_ONLY_PDFS = {"birthstone_guide.pdf"}
all_pdfs = sorted(DOCUMENTS_DIR.rglob("*.pdf"))
PDF_FILES = [path for path in all_pdfs if path.name not in TITLE_ONLY_PDFS]
if not PDF_FILES:
    raise FileNotFoundError(f"No usable PDFs found in {DOCUMENTS_DIR}")

USE_OCR_FALLBACK = which("tesseract") is not None
ENABLE_SEMANTIC_CHUNKING = False
RUN_RETRIEVAL_EVALUATION = True
RETRIEVAL_METHOD = "tfidf"
EVALUATION_DIR = PROJECT_ROOT / "data" / "rag" / "experiments" / "chunking_evaluation"
EVALUATION_CASES_FILE = EVALUATION_DIR / "document_chunking_cases.json"
EVALUATION_RESULTS_FILE = EVALUATION_DIR / "document_chunking_results.json"

print(f"SheCraft PDFs selected: {len(PDF_FILES)}/{len(all_pdfs)}")
for path in all_pdfs:
    if path.name in TITLE_ONLY_PDFS:
        print("Excluded title-only PDF:", path.relative_to(DOCUMENTS_DIR))
print("Extraction:", "native text with OCR fallback" if USE_OCR_FALLBACK else "native text (Tesseract unavailable)")

SheCraft PDFs selected: 39/40
Excluded title-only PDF: birthstone_guide.pdf
Extraction: native text (Tesseract unavailable)


# 2. Chunking strategies

Retrieval evaluation runs by default using the lightweight TF-IDF baseline. This keeps the retrieval method fixed while comparing chunking. The draft questions are provisional and cover every PDF with usable text.

In [2]:
STRATEGY_CONFIGS = {
    "fixed_size": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "recursive": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "sentence": {
        "enabled": True,
        "kwargs": {"sentences_per_chunk": 4, "sentence_overlap": 1},
    },
    "structure_aware": {
        "enabled": True,
        "kwargs": {"chunk_size": 800, "overlap": 100},
    },
    "semantic": {
        "enabled": ENABLE_SEMANTIC_CHUNKING,
        "kwargs": {
            "similarity_threshold": 0.65,
            "max_chunk_size": 800,
            "unit_prefix": "passage: ",
        },
    },
}

enabled_strategies = {
    name: config["kwargs"]
    for name, config in STRATEGY_CONFIGS.items()
    if config["enabled"]
}

print("Methods that will run:")
for name, settings in enabled_strategies.items():
    print(f" - {name}: {settings}")

Methods that will run:
 - fixed_size: {'chunk_size': 800, 'overlap': 100}
 - recursive: {'chunk_size': 800, 'overlap': 100}
 - sentence: {'sentences_per_chunk': 4, 'sentence_overlap': 1}
 - structure_aware: {'chunk_size': 800, 'overlap': 100}


## 3. Import extraction and chunking helpers

In [3]:
extraction_module = importlib.import_module("doc_rag.2_extraction")
chunking_module = importlib.import_module("doc_rag.4_chunking")
extract_pdf = extraction_module.extract_pdf
chunk_extracted_documents = chunking_module.chunk_extracted_documents

if RUN_RETRIEVAL_EVALUATION and not EVALUATION_CASES_FILE.is_file():
    raise FileNotFoundError(f"Evaluation cases were not found: {EVALUATION_CASES_FILE}")

if ENABLE_SEMANTIC_CHUNKING:
    try:
        from sentence_transformers import SentenceTransformer
    except ModuleNotFoundError as error:
        raise ModuleNotFoundError(
            f"Semantic chunking requires {error.name}. Install sentence-transformers in the SheCraft notebook kernel."
        ) from error
    embedding_model = SentenceTransformer("intfloat/multilingual-e5-base", device="cpu")
    embedding_model.max_seq_length = 512
else:
    embedding_model = None

if RUN_RETRIEVAL_EVALUATION:
    from sklearn.feature_extraction.text import TfidfVectorizer
    print("Retrieval evaluation: TF-IDF baseline (top 5)")


Retrieval evaluation: TF-IDF baseline (top 5)


# 4. Extract text from every selected PDF


In [4]:
extracted_documents = []

for position, pdf_file in enumerate(PDF_FILES, start=1):
    print(f"[{position}/{len(PDF_FILES)}] Extracting {pdf_file.name}")
    extracted_documents.append(extract_pdf(pdf_file, use_ocr_fallback=USE_OCR_FALLBACK))

extraction_summary = pd.DataFrame([
    {
        "file_name": document["file_name"],
        "pages": document["page_count"],
        "pages_with_text": sum(
            bool(page["final_text"].strip()) for page in document["pages"]
        ),
        "final_text_characters": sum(
            len(page["final_text"]) for page in document["pages"]
        ),
    }
    for document in extracted_documents
])
display(extraction_summary)

[1/39] Extracting engagement_ring_settings_guide.pdf


[2/39] Extracting alexandrite.pdf
[3/39] Extracting amber.pdf
[4/39] Extracting amethyst.pdf


[5/39] Extracting ametrine.pdf
[6/39] Extracting aquamarine.pdf
[7/39] Extracting citrine.pdf
[8/39] Extracting diamond.pdf


[9/39] Extracting emerald.pdf
[10/39] Extracting fancy_color_diamond.pdf
[11/39] Extracting garnet.pdf
[12/39] Extracting iolite.pdf
[13/39] Extracting jade.pdf


[14/39] Extracting kunzite.pdf
[15/39] Extracting lapis_lazuli.pdf
[16/39] Extracting moonstone.pdf
[17/39] Extracting morganite.pdf
[18/39] Extracting opal.pdf
[19/39] Extracting pearl.pdf


[20/39] Extracting peridot.pdf
[21/39] Extracting rose_quartz.pdf
[22/39] Extracting ruby.pdf
[23/39] Extracting sapphire.pdf
[24/39] Extracting spinel.pdf


... (5 outputs left out)

[36/39] Extracting pearl_care_cleaning_guide.pdf
[37/39] Extracting ring_metals_everyday_wear.pdf


[38/39] Extracting ring_size_guide.pdf


[39/39] Extracting ring_sizing_technical_guide.pdf


,file_name,pages,pages_with_text,final_text_characters
0,engagement_ring_settings_guide.pdf,15,14,19066
1,alexandrite.pdf,3,3,2422
2,amber.pdf,2,2,2524
3,amethyst.pdf,3,2,2604
4,ametrine.pdf,2,2,2207
5,aquamarine.pdf,2,2,2525
6,citrine.pdf,2,2,2557
7,diamond.pdf,2,2,2512
8,emerald.pdf,2,2,2599
9,fancy_color_diamond.pdf,3,3,2516


## 5. Generate chunks with every enabled method


In [5]:
chunks_by_strategy = {}

for strategy_name, strategy_kwargs in enabled_strategies.items():
    print(f"Chunking with {strategy_name}: {strategy_kwargs}")
    runtime_kwargs = dict(strategy_kwargs)
    if strategy_name == "semantic":
        runtime_kwargs["model"] = embedding_model

    chunks_by_strategy[strategy_name] = chunk_extracted_documents(
        extracted_documents,
        strategy=strategy_name,
        **runtime_kwargs,
    )
    print("  chunks created:", len(chunks_by_strategy[strategy_name]))

Chunking with fixed_size: {'chunk_size': 800, 'overlap': 100}
  chunks created: 301
Chunking with recursive: {'chunk_size': 800, 'overlap': 100}
  chunks created: 300
Chunking with sentence: {'sentences_per_chunk': 4, 'sentence_overlap': 1}
  chunks created: 1763
Chunking with structure_aware: {'chunk_size': 800, 'overlap': 100}


  chunks created: 318


## 6. Compare chunk-size distributions


In [6]:
comparison_rows = []

for strategy_name, chunks in chunks_by_strategy.items():
    lengths = chunks["character_count"]
    comparison_rows.append({
        "strategy": strategy_name,
        "chunks": len(chunks),
        "documents": chunks["file_name"].nunique(),
        "pages": chunks[["file_name", "page_number"]].drop_duplicates().shape[0],
        "min_chars": int(lengths.min()),
        "median_chars": int(lengths.median()),
        "mean_chars": round(float(lengths.mean()), 1),
        "p90_chars": int(lengths.quantile(0.90)),
        "max_chars": int(lengths.max()),
        "short_chunks_lt_100": int((lengths < 100).sum()),
        "short_chunk_percent": round(float((lengths < 100).mean() * 100), 1),
    })

strategy_comparison = pd.DataFrame(comparison_rows).set_index("strategy")
display(strategy_comparison)


,chunks,documents,pages,min_chars,median_chars,mean_chars,p90_chars,max_chars,short_chunks_lt_100,short_chunk_percent
strategy,,,,,,,,,,
fixed_size,301,39,175,11,675,570.4,800,800,2,0.7
recursive,300,39,175,11,679,560.6,792,799,3,1.0
sentence,1763,39,175,3,98,114.5,216,364,896,50.8
structure_aware,318,39,175,11,517,503.3,775,800,13,4.1


## 7. Inspect comparable chunk samples


In [7]:
SAMPLE_PAGES = [("ring_size_guide.pdf", 1), ("alexandrite.pdf", 1)]
SAMPLE_LIMIT_PER_STRATEGY = 2

for file_name, page_number in SAMPLE_PAGES:
    print(f"\n{file_name} - page {page_number}")
    for strategy_name, chunks in chunks_by_strategy.items():
        sample = chunks[
            (chunks["file_name"] == file_name)
            & (chunks["page_number"] == page_number)
        ].head(SAMPLE_LIMIT_PER_STRATEGY)
        print(f"\n{strategy_name}: {len(sample)} sample chunks")
        for row in sample.itertuples(index=False):
            print(f"[{row.chunk_id}] {row.character_count} characters")
            print(row.text)


ring_size_guide.pdf - page 1

fixed_size: 2 sample chunks
[ring_size_guide-57eb092d71bc-p0001-c000] 800 characters
How to Determine Ring Size: Tips and Ring
Size Chart
Getting the ring size right is a must for any engagement ring. After all, you’ve invested a lot in selecting the
perfect ring and planning the proposal, so naturally you’ll want the ring to fit right out of the box.
In this blog, we’ll dispel some common myths floating around on the Internet and provide some reliable
tips on how to determine her ring size, including an international ring size chart, so you can get it right.
Before you read any further, here’s a quick note on the importance of determining the right ring size. In
general, rings with pavé and channel settings cannot be resized easily. It also can be very difficult to resize
eternity/anniversary rings (that is, those with diamonds around the entire band). Examples of other r
[ring_size_guide-57eb092d71bc-p0001-c001] 800 characters
ternity/anniversary rings 

## 8. Integrity checks


In [8]:
expected_pages = {
    (document["source_path"], page["page_number"])
    for document in extracted_documents
    for page in document["pages"]
    if page["final_text"].strip()
}
empty_text_documents = [
    document["file_name"]
    for document in extracted_documents
    if not any(page["final_text"].strip() for page in document["pages"])
]
assert not empty_text_documents, f"Documents without usable text: {empty_text_documents}"

integrity_rows = []

for strategy_name, chunks in chunks_by_strategy.items():
    config = enabled_strategies[strategy_name]
    configured_limit = config.get("chunk_size", config.get("max_chunk_size"))
    limit_respected = (
        True
        if configured_limit is None
        else bool(chunks["character_count"].le(configured_limit).all())
    )
    chunk_pages = set(zip(chunks["source_path"], chunks["page_number"]))
    integrity_rows.append({
        "strategy": strategy_name,
        "all_text_pages_covered": chunk_pages == expected_pages,
        "nonempty_text": bool(chunks["text"].str.strip().ne("").all()),
        "unique_chunk_ids": bool(chunks["chunk_id"].is_unique),
        "valid_page_numbers": bool(chunks["page_number"].ge(1).all()),
        "source_files_exist": bool(
            chunks["source_path"].map(lambda value: Path(value).is_file()).all()
        ),
        "configured_size_limit_respected": limit_respected,
    })

integrity_checks = pd.DataFrame(integrity_rows).set_index("strategy")
display(integrity_checks)
assert integrity_checks.to_numpy().all(), "At least one chunk integrity check failed."
print("All integrity checks passed.")

,all_text_pages_covered,nonempty_text,unique_chunk_ids,valid_page_numbers,source_files_exist,configured_size_limit_respected
strategy,,,,,,
fixed_size,True,True,True,True,True,True
recursive,True,True,True,True,True,True
sentence,True,True,True,True,True,True
structure_aware,True,True,True,True,True,True


All integrity checks passed.


## 9. Retrieval evaluation

This stage needs the local embedding model and a SheCraft
`document_chunking_cases.json` file. Each case specifies an ID, a question,
the expected PDF, and one or more source pages. Results compare chunking
strategies by ranking their chunks with cosine similarity.

In [9]:
if RUN_RETRIEVAL_EVALUATION:
    with EVALUATION_CASES_FILE.open(encoding="utf-8") as file:
        evaluation_payload = json.load(file)

    evaluation_cases = evaluation_payload["cases"]
    if not evaluation_cases:
        raise ValueError("Add at least one SheCraft retrieval evaluation case.")
    available_pages = {
        (document["file_name"], page["page_number"])
        for document in extracted_documents
        for page in document["pages"]
        if page["final_text"].strip()
    }
    missing_labels = [
        (case["id"], case["expected_file"], page)
        for case in evaluation_cases
        for page in case["expected_pages"]
        if (case["expected_file"], page) not in available_pages
    ]
    if missing_labels:
        raise ValueError(f"Evaluation cases reference pages without extracted text: {missing_labels}")
    print("Evaluation cases loaded:", len(evaluation_cases))
    print("Distinct PDFs represented:", len({case["expected_file"] for case in evaluation_cases}))


Evaluation cases loaded: 39
Distinct PDFs represented: 39


In [10]:
def score_chunking(strategy_name, chunks, score_matrix, retrieval_method, *, positive_only=False):
    case_rows = []
    reciprocal_ranks = []
    hits_at_1, hits_at_3, hits_at_5 = [], [], []

    for query_index, case in enumerate(evaluation_cases):
        scores = score_matrix[query_index]
        candidates = np.flatnonzero(scores > 0) if positive_only else np.arange(len(scores))
        positions = candidates[np.argsort(-scores[candidates], kind="stable")][:5]
        expected_pages = set(case["expected_pages"])
        first_relevant_rank = None
        top_results = []
        for rank, position in enumerate(positions, start=1):
            row = chunks.iloc[int(position)]
            same_document = row["file_name"] == case["expected_file"]
            relevant = bool(same_document and int(row["page_number"]) in expected_pages)
            if relevant and first_relevant_rank is None:
                first_relevant_rank = rank
            top_results.append({
                "rank": rank,
                "chunk_id": row["chunk_id"],
                "file_name": row["file_name"],
                "page_number": int(row["page_number"]),
                "score": round(float(score_matrix[query_index, position]), 6),
                "relevant": relevant,
            })

        reciprocal_ranks.append(1.0 / first_relevant_rank if first_relevant_rank else 0.0)
        hits_at_1.append(first_relevant_rank is not None and first_relevant_rank <= 1)
        hits_at_3.append(first_relevant_rank is not None and first_relevant_rank <= 3)
        hits_at_5.append(first_relevant_rank is not None and first_relevant_rank <= 5)
        case_rows.append({
            "strategy": strategy_name,
            "retrieval_method": retrieval_method,
            "case_id": case["id"],
            "query": case["query"],
            "expected_file": case["expected_file"],
            "expected_pages": case["expected_pages"],
            "first_relevant_rank": first_relevant_rank,
            "top_5": top_results,
        })

    metrics = {
        "strategy": strategy_name,
        "retrieval_method": retrieval_method,
        "recall_at_1": round(float(np.mean(hits_at_1)), 4),
        "recall_at_3": round(float(np.mean(hits_at_3)), 4),
        "recall_at_5": round(float(np.mean(hits_at_5)), 4),
        "mrr_at_5": round(float(np.mean(reciprocal_ranks)), 4),
    }
    return metrics, case_rows

if RUN_RETRIEVAL_EVALUATION:
    retrieval_rows = []
    strategy_metric_rows = []

    for strategy_name, chunks in chunks_by_strategy.items():
        print(f"Evaluating {strategy_name} with TF-IDF")
        vectorizer = TfidfVectorizer(
            lowercase=True, strip_accents=None,
            token_pattern=r"(?u)\b\w\w+\b", ngram_range=(1, 2), sublinear_tf=True,
        )
        chunk_matrix = vectorizer.fit_transform(chunks["text"].astype(str))
        query_matrix = vectorizer.transform([case["query"] for case in evaluation_cases])
        score_matrix = (query_matrix @ chunk_matrix.T).toarray()
        metrics, case_rows = score_chunking(
            strategy_name, chunks, score_matrix, RETRIEVAL_METHOD, positive_only=True
        )
        strategy_metric_rows.append(metrics)
        retrieval_rows.extend(case_rows)

    retrieval_details = pd.DataFrame(retrieval_rows)
    retrieval_metrics = pd.DataFrame(strategy_metric_rows).set_index("strategy")
    display(retrieval_metrics.sort_values(["recall_at_5", "mrr_at_5"], ascending=False))


Evaluating fixed_size with TF-IDF


Evaluating recursive with TF-IDF

Evaluating sentence with TF-IDF


Evaluating structure_aware with TF-IDF


,retrieval_method,recall_at_1,recall_at_3,recall_at_5,mrr_at_5
strategy,,,,,
structure_aware,tfidf,0.6667,0.9231,1.0000,0.7915
fixed_size,tfidf,0.6154,0.9231,1.0000,0.7701
recursive,tfidf,0.5897,0.9231,1.0000,0.7628
sentence,tfidf,0.5385,0.8205,0.8462,0.6632


In [11]:
if RUN_RETRIEVAL_EVALUATION:
    failed_cases = retrieval_details[
        retrieval_details["first_relevant_rank"].isna()
    ]
    display(
        failed_cases[
            ["strategy", "case_id", "query", "expected_file", "expected_pages"]
        ]
    )

    results_payload = {
        "schema_version": 1,
        "evaluation_type": "provisional_notebook_technique_comparison",
        "status": "draft",
        "retrieval_method": RETRIEVAL_METHOD,
        "relevance": "expected PDF and page",
        "evaluation_case_file": str(
            EVALUATION_CASES_FILE.relative_to(PROJECT_ROOT)
        ),
        "strategy_configs": enabled_strategies,
        "retrieval_metrics": retrieval_metrics.reset_index().to_dict(
            orient="records"
        ),
        "chunk_statistics": strategy_comparison.reset_index().to_dict(
            orient="records"
        ),
        "case_results": retrieval_rows,
    }
    EVALUATION_RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with EVALUATION_RESULTS_FILE.open("w", encoding="utf-8") as file:
        json.dump(results_payload, file, ensure_ascii=False, indent=2)
    print("Saved provisional TF-IDF notebook evaluation to:", EVALUATION_RESULTS_FILE)
else:
    print("Chunking ran, but retrieval quality was not evaluated. Enable RUN_RETRIEVAL_EVALUATION and rerun all cells.")


,strategy,case_id,query,expected_file,expected_pages
79,sentence,gem_amber,"What is amber, and what kinds of inclusions ca...",amber.pdf,"[1, 2]"
80,sentence,gem_amethyst,"What is amethyst, what is its color, and how h...",amethyst.pdf,"[1, 2]"
88,sentence,gem_iolite,"What is iolite known for, including its color ...",iolite.pdf,"[1, 2]"
89,sentence,gem_jade,What minerals can the name jade refer to?,jade.pdf,[1]
104,sentence,gem_tourmaline,"What colors can tourmaline have, and which tra...",tourmaline.pdf,"[1, 2]"
111,sentence,bench_pave,"What is pavé setting, and what bench-work cons...",jewelry_settings_bench_guide.pdf,"[9, 10]"


Saved provisional TF-IDF notebook evaluation to: C:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\data\rag\experiments\chunking_evaluation\document_chunking_results.json


## 10. Recheck the leading chunking methods with E5-small-v2

Use the same cases and page labels. Compare these scores with the TF-IDF baseline above.

In [12]:
if RUN_RETRIEVAL_EVALUATION:
    import torch
    from sentence_transformers import SentenceTransformer

    dense_model_name = "intfloat/e5-small-v2"
    dense_device = "cuda" if torch.cuda.is_available() else "cpu"
    dense_model = SentenceTransformer(dense_model_name, device=dense_device)
    dense_embeddings = importlib.import_module("doc_rag.5_embeddings")
    query_vectors = dense_embeddings.embed_queries(
        [case["query"] for case in evaluation_cases],
        model=dense_model, model_name=dense_model_name, batch_size=8,
    )
    dense_metrics, dense_cases = [], []
    for strategy in ("fixed_size", "recursive", "structure_aware"):
        print("Evaluating with E5:", strategy)
        chunks = chunks_by_strategy[strategy]
        passages = dense_embeddings.prepare_texts(
            chunks["text"].tolist(), model_name=dense_model_name, text_type="passage"
        )
        chunk_vectors = dense_embeddings.embed_texts(passages, dense_model, batch_size=8)
        metrics, case_rows = score_chunking(
            strategy, chunks, query_vectors @ chunk_vectors.T, "dense_cosine"
        )
        dense_metrics.append(metrics)
        dense_cases.extend(case_rows)
    display(pd.DataFrame(dense_metrics).sort_values(
        ["recall_at_5", "mrr_at_5", "recall_at_1"], ascending=False
    ))
    dense_results_file = EVALUATION_RESULTS_FILE.with_name("dense_chunking_results.json")
    dense_payload = {
        **results_payload,
        "retrieval_method": "dense_cosine",
        "embedding_model": dense_model_name,
        "device": dense_device,
        "strategy_configs": {row["strategy"]: enabled_strategies[row["strategy"]] for row in dense_metrics},
        "retrieval_metrics": dense_metrics,
        "case_results": dense_cases,
    }
    dense_results_file.write_text(
        json.dumps(dense_payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Saved dense chunking results:", dense_results_file)


C:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\tmp\shecraft-notebooks\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  29%|██▊       | 57/199 [00:00<00:00, 551.96it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1804.75it/s]

Evaluating with E5: fixed_size


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

... (119 outputs left out)

Batches:  98%|█████████▊| 39/40 [00:31<00:00,  3.32it/s]

Batches: 100%|██████████| 40/40 [00:31<00:00,  1.26it/s]

,strategy,retrieval_method,recall_at_1,recall_at_3,recall_at_5,mrr_at_5
0,fixed_size,dense_cosine,0.9231,1.0000,1.0,0.9530
2,structure_aware,dense_cosine,0.8974,0.9744,1.0,0.9423
1,recursive,dense_cosine,0.8718,1.0000,1.0,0.9316


Saved dense chunking results: C:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\data\rag\experiments\chunking_evaluation\dense_chunking_results.json
